<a href="https://colab.research.google.com/github/b87512924-sys/widsproject/blob/main/widsproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [165]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("✅ All necessary libraries imported.")



def create_sample_rajasthan_data():
    """
    Creates sample solar data for Rajasthan with enhanced features.
    """
    np.random.seed(42)

    n_samples = 200
    latitudes = np.random.uniform(24.0, 30.0, n_samples)
    longitudes = np.random.uniform(69.0, 78.0, n_samples)
    land_cover_types = ['Agriculture', 'Forest', 'Built-up', 'Shrubland', 'Barren']

    data = []

    for i in range(n_samples):
        lat = latitudes[i]
        lon = longitudes[i]
        is_western = 1 if lon < 74 else 0

        elevation = np.random.uniform(100, 500)
        slope = np.random.exponential(2)
        ndvi = np.random.uniform(-0.1, 0.3)
        temperature = np.random.uniform(25, 45)
        cloud_cover = np.random.uniform(0.05, 0.3)

        aspect_deg = np.random.uniform(0, 360)
        hillshade = np.random.uniform(150, 255)
        bare_soil_index = np.random.uniform(-0.5, 0.8)
        brightness = np.random.uniform(0.1, 0.6)
        land_cover_type = np.random.choice(land_cover_types)

        solar_potential = (
            5.8 +  # Base for Rajasthan
            (0.2 if is_western else 0) +
            -0.0005 * elevation +
            -0.1 * slope +
            -0.3 * ndvi +
            -0.01 * temperature +
            -1.0 * cloud_cover +
            + 0.001 * np.cos(np.deg2rad(aspect_deg)) +
            + 0.005 * (hillshade - 150) / 100 +
            - 0.5 * bare_soil_index +
            + 0.5 * brightness +
            (0.1 if land_cover_type == 'Barren' else
             -0.1 if land_cover_type == 'Built-up' else
             0) +
            np.random.normal(0, 0.2)
        )
        solar_potential = np.clip(solar_potential, 4.5, 7.0)

        if solar_potential > 6.0:
            suitability = 'Excellent'
            color = 'green'
        elif solar_potential > 5.5:
            suitability = 'Good'
            color = 'blue'
        elif solar_potential > 5.0:
            suitability = 'Moderate'
            color = 'yellow'
        else:
            suitability = 'Poor'
            color = 'red'

        data.append({
            'latitude': lat,
            'longitude': lon,
            'predicted_solar_potential': solar_potential,
            'suitability': suitability,
            'color': color,
            'region': 'Rajasthan',
            'elevation_m': elevation,
            'slope_deg': slope,
            'ndvi': ndvi,
            'temperature_c': temperature,
            'cloud_cover_perc': cloud_cover,
            'aspect_deg': aspect_deg,
            'hillshade': hillshade,
            'bare_soil_index': bare_soil_index,
            'brightness': brightness,
            'land_cover_type': land_cover_type
        })

    return pd.DataFrame(data)

print("✅ `create_sample_rajasthan_data` function defined.")


def prepare_ml_data(df):
    """
    Prepare features and target for regression model.
    Converts raw data into an ML-ready format by selecting features,
    handling categorical variables, scaling numerical features, and splitting data.
    """
    print("\n⚙️ PREPARING DATA FOR MACHINE LEARNING")
    print("="*50)

    feature_cols = [
        'elevation_m',
        'slope_deg',
        'ndvi',
        'temperature_c',
        'cloud_cover_perc',
        'aspect_deg',
        'hillshade',
        'bare_soil_index',
        'brightness'
    ]

    print(f"Selected {len(feature_cols)} numerical features:")
    for feature in feature_cols:
        print(f"  • {feature}")

    target_col = 'predicted_solar_potential'
    print(f"\nTarget variable: {target_col}")

    missing_features = [col for col in feature_cols if col not in df.columns]
    if missing_features:
        print(f"\n⚠️ Warning: Missing feature columns: {missing_features}. They will be skipped.")
        feature_cols = [col for col in feature_cols if col in df.columns]

    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in the DataFrame.")

    categorical_cols = ['region', 'land_cover_type']
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    encoded_feature_cols = [col for col in df_encoded.columns if col.startswith('region_') or col.startswith('land_cover_type_')]
    final_feature_cols = feature_cols + encoded_feature_cols

    X = df_encoded[final_feature_cols]
    y = df_encoded[target_col]

    print(f"\n📐 Data dimensions:")
    print(f"X shape: {X.shape} ({X.shape[0]} samples × {X.shape[1]} features)")
    print(f"y shape: {y.shape}")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nSplitting data into training (80%) and testing (20%) sets:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    numerical_cols_to_scale = [col for col in feature_cols if col in X_train.columns]

    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    if numerical_cols_to_scale:
        print("\nScaling numerical features...")
        X_train_scaled[numerical_cols_to_scale] = scaler.fit_transform(X_train[numerical_cols_to_scale])
        X_test_scaled[numerical_cols_to_scale] = scaler.transform(X_test[numerical_cols_to_scale])
        print("✅ Numerical features scaled successfully.")
    else:
        print("\nNo numerical features to scale or already scaled.")

    print("\n✅ Data preparation complete!")
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, final_feature_cols

print("✅ `prepare_ml_data` function defined.")


def train_regression_model(X_train, X_test, y_train, y_test, feature_cols):
    """
    Train a supervised regression model (Linear Regression) to predict solar potential
    """
    print("\n🤖 TRAINING REGRESSION MODEL")
    print("="*50)

    print("Model: Linear Regression")
    print("Purpose: Predict solar potential (GHI) based on environmental features")

    model = LinearRegression()

    print("\n🚀 Training model...")
    model.fit(X_train, y_train)
    print("✅ Model trained successfully!")

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    def calculate_metrics(y_true, y_pred, dataset_name):
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100 if np.all(y_true != 0) else np.nan

        return {
            'Dataset': dataset_name,
            'R² Score': r2,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE (%)': mape
        }

    train_metrics = calculate_metrics(y_train, y_train_pred, 'Training')
    test_metrics = calculate_metrics(y_test, y_test_pred, 'Testing')

    metrics_df = pd.DataFrame([train_metrics, test_metrics])
    print("\n📊 MODEL PERFORMANCE METRICS:")
    display(metrics_df)

    print("\n🔄 Performing cross-validation (5-fold)...")
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    print(f"  Cross-validation R² Scores: {cv_scores}")
    print(f"  Mean CV R² Score: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
    print("✅ Cross-validation complete!")

    return model, metrics_df, cv_scores

print("✅ `train_regression_model` function defined.")


print("\n--- Starting Full Execution Flow ---")

# 1. Create sample Rajasthan solar data
rajasthan_solar_data = create_sample_rajasthan_data()
print("\nSample Rajasthan solar data created successfully with new features:")
display(rajasthan_solar_data.head())

# 2. Prepare ML data
X_train, X_test, y_train, y_test, scaler, ml_feature_cols = prepare_ml_data(rajasthan_solar_data)
print("\nPrepared dataframes and scaler are available.")

# 3. Train and evaluate the Linear Regression model
trained_model_linear, model_metrics_linear, cv_scores_linear = train_regression_model(X_train, X_test, y_train, y_test, ml_feature_cols)

print("\n--- Full Execution Flow Complete --- ")
print("Trained model is available as 'trained_model_linear'.")
print("Performance metrics are available as 'model_metrics_linear'.")

✅ All necessary libraries imported.
✅ `create_sample_rajasthan_data` function defined.
✅ `prepare_ml_data` function defined.
✅ `train_regression_model` function defined.

--- Starting Full Execution Flow ---

Sample Rajasthan solar data created successfully with new features:


,latitude,longitude,predicted_solar_potential,suitability,color,region,elevation_m,slope_deg,ndvi,temperature_c,cloud_cover_perc,aspect_deg,hillshade,bare_soil_index,brightness,land_cover_type
0,26.247241,74.778285,5.288297,Moderate,yellow,Rajasthan,141.249548,4.656891,0.102101,41.529149,0.130012,322.388362,190.866176,-0.485911,0.552691,Shrubland
1,29.704286,69.757260,5.689620,Good,blue,Rajasthan,370.707962,1.317572,0.097210,26.665688,0.072926,216.878733,208.138820,-0.223454,0.573097,Agriculture
2,28.391964,70.454658,5.001644,Moderate,yellow,Rajasthan,145.385841,5.345244,0.289699,44.918625,0.063968,265.332800,207.321156,0.417581,0.584326,Agriculture
3,27.591951,77.086988,6.001318,Excellent,green,Rajasthan,140.449070,0.175711,0.180388,26.455260,0.255465,254.247202,158.541622,-0.389711,0.593320,Barren
4,24.936112,74.457862,4.500000,Poor,red,Rajasthan,248.256859,3.351151,0.278899,44.720021,0.238345,135.453451,158.767575,0.510291,0.379202,Built-up



⚙️ PREPARING DATA FOR MACHINE LEARNING
Selected 9 numerical features:
  • elevation_m
  • slope_deg
  • ndvi
  • temperature_c
  • cloud_cover_perc
  • aspect_deg
  • hillshade
  • bare_soil_index
  • brightness

Target variable: predicted_solar_potential

📐 Data dimensions:
X shape: (200, 13) (200 samples × 13 features)
y shape: (200,)

Splitting data into training (80%) and testing (20%) sets:
X_train shape: (160, 13)
X_test shape: (40, 13)
y_train shape: (160,)
y_test shape: (40,)

Scaling numerical features...
✅ Numerical features scaled successfully.

✅ Data preparation complete!

Prepared dataframes and scaler are available.

🤖 TRAINING REGRESSION MODEL
Model: Linear Regression
Purpose: Predict solar potential (GHI) based on environmental features

🚀 Training model...
✅ Model trained successfully!

📊 MODEL PERFORMANCE METRICS:


,Dataset,R² Score,RMSE,MAE,MAPE (%)
0,Training,0.653753,0.208081,0.159278,3.094075
1,Testing,0.616572,0.246775,0.206222,4.011418



🔄 Performing cross-validation (5-fold)...
  Cross-validation R² Scores: [0.43552289 0.641337   0.43146269 0.68869067 0.57985662]
  Mean CV R² Score: 0.5554 (+/- 0.1053)
✅ Cross-validation complete!

--- Full Execution Flow Complete --- 
Trained model is available as 'trained_model_linear'.
Performance metrics are available as 'model_metrics_linear'.


# Task
Set up Google Earth Engine and import the required Python libraries ('earthengine-api', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn') to prepare for a solar energy hotspot mapping project.

Now, let's generate the interactive solar map using our `rajasthan_solar_data`.

Now, let's execute the `perform_eda` function with our `rajasthan_solar_data` to generate the insights and visualizations.

In [186]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def perform_eda(df):
    print("🔍 PERFORMING EXPLORATORY DATA ANALYSIS")
    print("="*50)
    print("\n📊 BASIC INFORMATION:")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\n🔍 MISSING VALUES:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("✅ No missing values found!")
    else:
        print(missing[missing > 0])
    print("\n📈 STATISTICAL SUMMARY:")
    display(df.describe())
    fig, axes = plt.subplots(3, 2, figsize=(18, 20))
    fig.suptitle('Exploratory Data Analysis of Solar Hotspot Data', fontsize=20)
    sns.histplot(df['predicted_solar_potential'], kde=True, bins=20, color='skyblue', ax=axes[0, 0])
    axes[0, 0].axvline(x=5.5, color='orange', linestyle='--', label='Good threshold')
    axes[0, 0].axvline(x=6.0, color='green', linestyle='--', label='Excellent threshold')
    axes[0, 0].set_xlabel('Solar Potential (kWh/m²/day)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Solar Potential')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    suitability_counts = df['suitability'].value_counts(sort=False)
    sns.barplot(x=suitability_counts.index, y=suitability_counts.values,
                palette=['green', 'blue', 'yellow', 'red'], ax=axes[0, 1])
    axes[0, 1].set_xlabel('Suitability')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Suitability Distribution')
    axes[0, 1].tick_params(axis='x', rotation=45)
    numeric_df = df.select_dtypes(include=np.number)
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1, 0])
    axes[1, 0].set_title('Correlation Matrix of Numerical Features')
    sns.scatterplot(x='elevation_m', y='predicted_solar_potential', hue='suitability', data=df,
                    palette={'Excellent': 'green', 'Good': 'blue', 'Moderate': 'yellow', 'Poor': 'red'},
                    ax=axes[1, 1], alpha=0.7)
    axes[1, 1].set_title('Elevation vs. Predicted Solar Potential')
    axes[1, 1].set_xlabel('Elevation (m)')
    axes[1, 1].set_ylabel('Predicted Solar Potential (kWh/m²/day)')
    axes[1, 1].grid(True, alpha=0.3)
    sns.scatterplot(x='cloud_cover_perc', y='predicted_solar_potential', hue='suitability', data=df,
                    palette={'Excellent': 'green', 'Good': 'blue', 'Moderate': 'yellow', 'Poor': 'red'},
                    ax=axes[2, 0], alpha=0.7)
    axes[2, 0].set_title('Cloud Cover vs. Predicted Solar Potential')
    axes[2, 0].set_xlabel('Cloud Cover Percentage')
    axes[2, 0].set_ylabel('Predicted Solar Potential (kWh/m²/day)')
    axes[2, 0].grid(True, alpha=0.3)
    sns.scatterplot(x='ndvi', y='predicted_solar_potential', hue='suitability', data=df,
                    palette={'Excellent': 'green', 'Good': 'blue', 'Moderate': 'yellow', 'Poor': 'red'},
                    ax=axes[2, 1], alpha=0.7)
    axes[2, 1].set_title('NDVI vs. Predicted Solar Potential')
    axes[2, 1].set_xlabel('NDVI (Normalized Difference Vegetation Index)')
    axes[2, 1].set_ylabel('Predicted Solar Potential (kWh/m²/day)')
    axes[2, 1].grid(True, alpha=0.3)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

print("✅ `perform_eda` function defined.")